# Clasificación de Piso en el Dataset UJIIndoorLoc

---

## Introducción

En este notebook se implementa un flujo completo de procesamiento y análisis para la clasificación del **piso** en un entorno interior utilizando el dataset **UJIIndoorLoc**. Este conjunto de datos contiene mediciones de señales WiFi recopiladas en distintas ubicaciones de un edificio, con información sobre coordenadas, piso, usuario, hora, entre otros.

En esta tarea nos enfocaremos en predecir el **piso** en el que se encuentra un dispositivo, considerando únicamente las muestras etiquetadas con valores válidos para dicha variable. Se tratará como un problema de clasificación multiclase (planta baja, primer piso, segundo piso).

## Objetivos

- **Cargar y explorar** el conjunto de datos UJIIndoorLoc.
- **Preparar** los datos seleccionando las características relevantes y el target (`FLOOR`).
- **Dividir** el dataset en entrenamiento y validación (80/20).
- **Entrenar y optimizar** clasificadores basados en seis algoritmos:
  - K-Nearest Neighbors (KNN)
  - Gaussian Naive Bayes
  - Regresión Logística
  - Árboles de Decisión
  - Support Vector Machines (SVM)
  - Random Forest
- **Seleccionar hiperparámetros óptimos** para cada modelo utilizando validación cruzada (5-fold), empleando estrategias como **Grid Search**, **Randomized Search**, o **Bayesian Optimization** según el algoritmo.
- **Comparar el desempeño** de los modelos sobre el conjunto de validación, usando métricas como *accuracy*, *precision*, *recall*, y *F1-score*.
- **Determinar el mejor clasificador** para esta tarea, junto con sus hiperparámetros óptimos.

Este ejercicio permite no solo evaluar la capacidad predictiva de distintos algoritmos clásicos de clasificación, sino también desarrollar buenas prácticas en validación de modelos y selección de hiperparámetros en contextos del mundo real.

---

## Paso 1: Cargar y explorar el dataset

**Instrucciones:**
- Descarga el dataset **UJIIndoorLoc** desde la UCI Machine Learning Repository o utiliza la versión proporcionada en el repositorio del curso (por ejemplo: `datasets\UJIIndoorLoc\trainingData.csv`).
- Carga el dataset utilizando `pandas`.
- Muestra las primeras filas del dataset utilizando `df.head()`.
- Imprime el número total de muestras (filas) y características (columnas).
- Verifica cuántas clases distintas hay en la variable objetivo `FLOOR` y cuántas muestras tiene cada clase (`df['FLOOR'].value_counts()`).


In [31]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv("trainingData.csv")

# Mostrar primeras filas
print("Primeras filas del dataset:")
print(df.head())

# Número de filas y columnas
print("\nNúmero de muestras (filas):", df.shape[0])
print("Número de características (columnas):", df.shape[1])

# Distribución de la variable FLOOR
print("\nDistribución de clases en FLOOR:")
print(df['FLOOR'].value_counts())

# Número de clases distintas
print("\nNúmero de clases distintas en FLOOR:", df['FLOOR'].nunique())

Primeras filas del dataset:
   WAP001  WAP002  WAP003  WAP004  WAP005  WAP006  WAP007  WAP008  WAP009  \
0     100     100     100     100     100     100     100     100     100   
1     100     100     100     100     100     100     100     100     100   
2     100     100     100     100     100     100     100     -97     100   
3     100     100     100     100     100     100     100     100     100   
4     100     100     100     100     100     100     100     100     100   

   WAP010  ...  WAP520  LONGITUDE      LATITUDE  FLOOR  BUILDINGID  SPACEID  \
0     100  ...     100 -7541.2643  4.864921e+06      2           1      106   
1     100  ...     100 -7536.6212  4.864934e+06      2           1      106   
2     100  ...     100 -7519.1524  4.864950e+06      2           1      103   
3     100  ...     100 -7524.5704  4.864934e+06      2           1      102   
4     100  ...     100 -7632.1436  4.864982e+06      0           0      122   

   RELATIVEPOSITION  USERID  PHONE

---

## Paso 2: Preparar los datos

**Instrucciones:**

- Elimina las columnas que no son relevantes para la tarea de clasificación del piso:
  - `LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`
- Conserva únicamente:
  - Las columnas `WAP001` a `WAP520` como características (RSSI de puntos de acceso WiFi).
  - La columna `FLOOR` como variable objetivo.
- Verifica si existen valores atípicos o valores inválidos en las señales WiFi (por ejemplo: valores constantes como 100 o -110 que suelen indicar ausencia de señal).
- Separa el conjunto de datos en:
  - `X`: matriz de características (todas las columnas `WAP`)
  - `y`: vector objetivo (`FLOOR`)


In [32]:
# Eliminar columnas no relevantes
cols_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
                'USERID', 'PHONEID', 'TIMESTAMP']

df = df.drop(columns=cols_to_drop)

# Verificar que solo queden columnas WAP + FLOOR
print("Columnas actuales:")
print(df.columns)

# Verificar valores atípicos o inválidos en señales WiFi
print("\nVerificando valores especiales en el dataset:")

val_100 = (df == 100).sum().sum()
val_neg110 = (df == -110).sum().sum()

print("Cantidad de valores 100:", val_100)
print("Cantidad de valores -110:", val_neg110)

# Separar variables
X = df.drop(columns=['FLOOR'])  # características (WAPs)
y = df['FLOOR']                 # variable objetivo

# Mostrar dimensiones
print("\nDimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

Columnas actuales:
Index(['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005', 'WAP006', 'WAP007',
       'WAP008', 'WAP009', 'WAP010',
       ...
       'WAP513', 'WAP514', 'WAP515', 'WAP516', 'WAP517', 'WAP518', 'WAP519',
       'WAP520', 'FLOOR', 'BUILDINGID'],
      dtype='object', length=522)

Verificando valores especiales en el dataset:
Cantidad de valores 100: 10008477
Cantidad de valores -110: 0

Dimensiones de X: (19937, 521)
Dimensiones de y: (19937,)


---

## Paso 3: Preprocesamiento de las señales WiFi

**Contexto:**

Las columnas `WAP001` a `WAP520` representan la intensidad de la señal (RSSI) recibida desde distintos puntos de acceso WiFi. Los valores típicos de RSSI están en una escala negativa, donde:

- Valores cercanos a **0 dBm** indican señal fuerte.
- Valores cercanos a **-100 dBm** indican señal débil o casi ausente.
- Un valor de **100** en este dataset representa una señal **no detectada**, es decir, el punto de acceso no fue visto por el dispositivo en ese instante.

**Instrucciones:**

- Para facilitar el procesamiento y tratar la ausencia de señal de forma coherente, se recomienda mapear todos los valores **100** a **-100**, que semánticamente representa *ausencia de señal detectable*.
- Esto unifica el rango de valores y evita que 100 (un valor artificial) afecte negativamente la escala de los algoritmos.

**Pasos sugeridos:**

- Reemplaza todos los valores `100` por `-100` en las columnas `WAP001` a `WAP520`:
  ```python
  X[X == 100] = -100


In [33]:
# Reemplazar valores 100 por -100 en las señales WiFi

print("Valores 100 antes del reemplazo:", (X == 100).sum().sum())

# Reemplazo
X[X == 100] = -100

print("Valores 100 después del reemplazo:", (X == 100).sum().sum())

# Verificación adicional (opcional)
print("Valores -100 ahora presentes:", (X == -100).sum().sum())

Valores 100 antes del reemplazo: 10008477
Valores 100 después del reemplazo: 0
Valores -100 ahora presentes: 10008716


---

## Paso 4: Entrenamiento y optimización de hiperparámetros

**Objetivo:**

Entrenar y comparar distintos clasificadores para predecir correctamente el piso (`FLOOR`) y encontrar los mejores hiperparámetros para cada uno mediante validación cruzada.

**Clasificadores a evaluar:**

- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes
- Regresión Logística
- Árboles de Decisión
- Support Vector Machines (SVM)
- Random Forest

**Procedimiento:**

1. Divide el dataset en conjunto de **entrenamiento** (80%) y **validación** (20%) usando `train_test_split` con `stratify=y`.
2. Para cada clasificador:
   - Define el espacio de búsqueda de hiperparámetros.
   - Usa **validación cruzada 5-fold** sobre el conjunto de entrenamiento para seleccionar los mejores hiperparámetros.
   - Emplea una estrategia de búsqueda adecuada:
     - **GridSearchCV**: búsqueda exhaustiva (ideal para espacios pequeños).
     - **RandomizedSearchCV**: búsqueda aleatoria (más eficiente con espacios amplios).
     - **Bayesian Optimization** (opcional): para búsquedas más inteligentes, usando librerías como `optuna` o `skopt`.
3. Guarda el mejor modelo encontrado para cada clasificador con su configuración óptima.



In [34]:
# create the training and validation sets

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

#  División de datos (80% entrenamiento - 20% validación)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape)
print("Validación:", X_val.shape)

best_models = {}


Train: (15949, 521)
Validación: (3988, 521)


In [35]:
# train and optimize KNN

knn = KNeighborsClassifier()
params_knn = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance']
}
grid_knn = GridSearchCV(knn, params_knn, cv=5)
grid_knn.fit(X_train, y_train)

best_models['KNN'] = grid_knn.best_estimator_
print("KNN mejores parámetros:", grid_knn.best_params_)


KNN mejores parámetros: {'n_neighbors': 3, 'weights': 'distance'}


In [36]:
# train and optimize Gaussian Naive Bayes

gnb = GaussianNB()
gnb.fit(X_train, y_train)

best_models['Naive Bayes'] = gnb
print("Naive Bayes: sin hiperparámetros (default)")


Naive Bayes: sin hiperparámetros (default)


In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Definir modelo (más eficiente)
logreg = LogisticRegression(max_iter=500, solver='liblinear')

param_grid = {
    'C': [0.1, 1, 10]
}

# Grid Search con validación cruzada
grid_logreg = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

# Entrenar
grid_logreg.fit(X_train, y_train)

# Guardar mejor modelo
best_logreg = grid_logreg.best_estimator_

print("Mejores parámetros:", grid_logreg.best_params_)
print("Mejor accuracy (CV):", grid_logreg.best_score_)

Mejores parámetros: {'C': 0.1}
Mejor accuracy (CV): 0.9905950687298134


In [38]:
# train and optimize decision tree

tree = DecisionTreeClassifier()
params_tree = {
    'max_depth': [10, 20, None],
    'criterion': ['gini', 'entropy']
}
grid_tree = GridSearchCV(tree, params_tree, cv=5)
grid_tree.fit(X_train, y_train)

best_models['Decision Tree'] = grid_tree.best_estimator_
print("Decision Tree mejores parámetros:", grid_tree.best_params_)

Decision Tree mejores parámetros: {'criterion': 'gini', 'max_depth': None}


In [39]:
# train and optimize Support Vector Machine

svm = SVC()
params_svm = {
    'C': [1, 10],
    'kernel': ['rbf']
}
grid_svm = GridSearchCV(svm, params_svm, cv=5)
grid_svm.fit(X_train, y_train)

best_models['SVM'] = grid_svm.best_estimator_
print("SVM mejores parámetros:", grid_svm.best_params_)


SVM mejores parámetros: {'C': 10, 'kernel': 'rbf'}


In [40]:
# train and optimize Random Forest

rf = RandomForestClassifier()
params_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}
grid_rf = GridSearchCV(rf, params_rf, cv=5)
grid_rf.fit(X_train, y_train)

best_models['Random Forest'] = grid_rf.best_estimator_
print("Random Forest mejores parámetros:", grid_rf.best_params_)

Random Forest mejores parámetros: {'max_depth': None, 'n_estimators': 200}


---

## Paso 5: Crear una tabla resumen de los mejores modelos

**Instrucciones:**

Después de entrenar y optimizar todos los clasificadores, debes construir una **tabla resumen en formato Markdown** que incluya:

- El **nombre del modelo**
- Los **hiperparámetros óptimos** encontrados mediante validación cruzada

### Requisitos:

- La tabla debe estar escrita en formato **Markdown**.
- Cada fila debe corresponder a uno de los modelos evaluados.
- Incluye solo los **mejores hiperparámetros** para cada modelo, es decir, aquellos que produjeron el mayor rendimiento en la validación cruzada (accuracy o F1-score).
- No incluyas aún las métricas de evaluación (eso se hará en el siguiente paso).

### Ejemplo de formato:


| Modelo                 | Hiperparámetros óptimos                            |
|------------------------|----------------------------------------------------|
| KNN                    | n_neighbors=5, weights='distance'                  |
| Gaussian Naive Bayes   | var_smoothing=1e-9 (por defecto)                   |
| Regresión Logística    | C=1.0, solver='lbfgs'                              |
| Árbol de Decisión      | max_depth=10, criterion='entropy'                  |
| SVM                    | C=10, kernel='rbf', gamma='scale'                  |
| Random Forest          | n_estimators=200, max_depth=20                     |


# tu tabla de resultados aquí

| Modelo               | Hiperparámetros óptimos |
|----------------------|------------------------|
| KNN                  | n_neighbors=3, weights='distance' |
| Gaussian Naive Bayes | Default |
| Regresión Logística  | C=0.1 |
| Árbol de Decisión    | max_depth=None, criterion='gini' |
| SVM                  | C=10, kernel='rbf' |
| Random Forest        | n_estimators=200, max_depth=None |

---

## Paso 6: Preparar los datos finales para evaluación

**Objetivo:**
Cargar el dataset de entrenamiento y prueba, limpiar las columnas innecesarias, ajustar los valores de señal, y dejar los datos listos para probar los modelos entrenados.

**Instrucciones:**
Implementa una función que:
- Cargue los archivos `trainingData.csv` y `validationData.csv`
- Elimine las columnas irrelevantes (`LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`)
- Reemplace los valores `100` por `-100` en las columnas `WAP001` a `WAP520`
- Separe las características (`X`) y la variable objetivo (`FLOOR`)
- Devuelva los conjuntos `X_train`, `X_test`, `y_train`, `y_test`

In [41]:
import pandas as pd

def load_data(train_path, test_path):

    # Columnas a eliminar
    cols_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION',
                    'USERID', 'PHONEID', 'TIMESTAMP']

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    train = train.drop(columns=cols_to_drop)
    test = test.drop(columns=cols_to_drop)

    X_train = train.drop(columns=['FLOOR'])
    y_train = train['FLOOR']

    X_test = test.drop(columns=['FLOOR'])
    y_test = test['FLOOR']

    X_train[X_train == 100] = -100
    X_test[X_test == 100] = -100

    return X_train, X_test, y_train, y_test

In [42]:
X_train, X_test, y_train, y_test = load_data(
    "trainingData.csv",
    "validationData.csv"
)

print(X_train.shape, X_test.shape)

(19937, 521) (1111, 521)


---

## Paso 7: Evaluar modelos optimizados en el conjunto de prueba

**Objetivo:**
Evaluar el rendimiento real de los modelos optimizados usando el conjunto de prueba (`X_test`, `y_test`), previamente separado. Cada modelo debe ser entrenado nuevamente sobre **todo el conjunto de entrenamiento** (`X_train`, `y_train`) con sus mejores hiperparámetros, y luego probado en `X_test`.

**Instrucciones:**

1. Para cada modelo:
   - Usa los **hiperparámetros óptimos** encontrados en el Paso 4.
   - Entrena el modelo con `X_train` y `y_train`.
   - Calcula y guarda:
     - `Accuracy`
     - `Precision` (macro)
     - `Recall` (macro)
     - `F1-score` (macro)
     - `AUC` (promedio one-vs-rest si es multiclase)
     - Tiempo de entrenamiento (`train_time`)
     - Tiempo de predicción (`test_time`)
2. Muestra todos los resultados en una **tabla comparativa**


In [43]:
import time
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Modelos con tus mejores parámetros
models = {
    "KNN": KNeighborsClassifier(n_neighbors=3, weights='distance'),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(C=0.1, solver='liblinear', max_iter=300),
    "Decision Tree": DecisionTreeClassifier(max_depth=None, criterion='gini'),
    "SVM": SVC(C=10, kernel='rbf', probability=True),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=None)
}

# Binarizar etiquetas para AUC
classes = sorted(y_train.unique())
y_test_bin = label_binarize(y_test, classes=classes)

results = []

for name, model in models.items():

    # Tiempo entrenamiento
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train

    # Tiempo predicción
    start_test = time.time()
    y_pred = model.predict(X_test)
    test_time = time.time() - start_test

    # Métricas básicas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # AUC (manejo seguro de errores)
    auc = "No disponible"
    if hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X_test)
            auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
        except:
            auc = "Error"

    results.append([
        name, accuracy, precision, recall, f1, auc, train_time, test_time
    ])

# Crear tabla
results_df = pd.DataFrame(results, columns=[
    "Modelo", "Accuracy", "Precision", "Recall", "F1-score", "AUC",
    "Train Time (s)", "Test Time (s)"
])

print(results_df)

                Modelo  Accuracy  Precision    Recall  F1-score       AUC  \
0                  KNN  0.907291   0.921995  0.900514  0.908401  0.946329   
1          Naive Bayes  0.474347   0.482471  0.588766  0.460280  0.780224   
2  Logistic Regression  0.891089   0.872364  0.897598  0.882963  0.970370   
3        Decision Tree  0.803780   0.807541  0.800879  0.800957  0.874670   
4                  SVM  0.914491   0.897685  0.915372  0.904757  0.981769   
5        Random Forest  0.912691   0.921666  0.887994  0.900333  0.986345   

   Train Time (s)  Test Time (s)  
0        0.359364       1.668125  
1        0.173966       0.016546  
2       31.225562       0.007046  
3        1.270445       0.004844  
4       50.759279       0.779732  
5       11.199429       0.073014  


---
## Paso 8: Selección y justificación del mejor modelo

**Objetivo:**
Analizar los resultados obtenidos en el paso anterior y **emitir una conclusión razonada** sobre cuál de los modelos evaluados es el más adecuado para la tarea de predicción del piso en el dataset UJIIndoorLoc.

**Instrucciones:**

- Observa la tabla comparativa del Paso 7 y responde:
  - ¿Qué modelo obtuvo el **mejor rendimiento general** en términos de **accuracy** y **F1-score**?
  - ¿Qué tan consistente fue su rendimiento en **precision** y **recall**?
  - ¿Tiene un **tiempo de entrenamiento o inferencia** excesivamente alto?
  - ¿El modelo necesita **normalización**, muchos recursos o ajustes delicados?
- Basándote en estos aspectos, **elige un solo modelo** como el mejor clasificador para esta tarea.
- **Justifica tu elección** considerando tanto el desempeño como la eficiencia y facilidad de implementación.


# tu respuesta aquí

Aunque el modelo SVM presenta el mejor rendimiento en términos de accuracy y F1-score, su tiempo de entrenamiento es considerablemente mayor en comparación con otros modelos.

En modelos como KNN y Random Forest ofrecen un rendimiento competitivo, con valores cercanos en las métricas principales, pero con tiempos de entrenamiento significativamente menores, especialmente en el caso de KNN.

Esto indica que, SVM es el modelo más preciso, y el KNN podría ser una mejor alternativa en escenarios donde la eficiencia computacional y la rapidez sean factores críticos.

Por lo tanto para esta tarea específica, donde se prioriza la precisión en la predicción del piso, se selecciona SVM como el mejor modelo.

---

## Rúbrica de Evaluación

| Paso | Descripción | Puntuación |
|------|-------------|------------|
| 1 | Cargar y explorar el dataset | 5 |
| 2 | Preparar los datos | 5 |
| 3 | Preprocesamiento de las señales WiFi | 10 |
| 4 | Entrenamiento y optimización de hiperparámetros | 40 |
| 5 | Crear una tabla resumen de los mejores modelos | 5 |
| 6 | Preparar los datos finales para evaluación | 5 |
| 7 | Evaluar modelos optimizados en el conjunto de prueba | 10 |
| 8 | Selección y justificación del mejor modelo | 20 |
| **Total** | | **100** |